# Week 74, Day 2: Code review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w074d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import difflib

before = ["def total(prices):", "    t = 0", "    for p in prices:", "        t += p", "    return t", ""]
after = ["def total(prices, tax=0.0):", '    """The total, with tax added."""', "    return sum(prices) * (1 + tax)", ""]
for line in difflib.unified_diff(before, after, "a/shop.py", "b/shop.py", lineterm=""):
    print(line)

> **Key idea.** A unified diff has ---/+++ file headers and hunks @@ -old_start,count +new_start,count @@; context lines advance both line numbers, - lines the old, + lines the new; a missing count means 1. Review top down: purpose, design, correctness, tests, security, readability, then details; leave mechanical checks to tools; keep changes under a few hundred lines. Comments are kind, about the code, specific, actionable, questions when unsure, and labelled (issue, suggestion, question, nitpick, praise; blocking or not). Authors keep pull requests small and described, review their own diff first, get checks green, and answer every comment.

### Exercise: Parsing a unified diff

Write `parse_diff(text)`: return a list of files, each a dict with `"old"` and `"new"` (the paths after `--- ` and `+++ `, with any leading `a/` or `b/` removed; keep `/dev/null` as it is) and `"hunks"`: a list of dicts with `"old_start"`, `"old_count"`, `"new_start"`, `"new_count"` (a missing count is 1) and `"lines"`: the hunk's lines as `(kind, text)` pairs, where kind is `" "`, `"-"` or `"+"` and text is the rest of the line. Ignore lines before the first `---` (such as `diff --git` and `index` lines) and `\ No newline at end of file` lines.

In [ ]:
import re

def parse_diff(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse():\n    \"files, hunks and lines\"\n    diff = \"\"\"diff --git a/shop.py b/shop.py\nindex 1a2b..3c4d 100644\n--- a/shop.py\n+++ b/shop.py\n@@ -1,5 +1,3 @@\n-def total(prices):\n-    t = 0\n-    for p in prices:\n-        t += p\n-    return t\n+def total(prices, tax=0.0):\n+    return sum(prices) * (1 + tax)\n+\n@@ -20 +18,2 @@ class Cart:\n-    pass\n+    def clear(self):\n+        self.items = []\n--- /dev/null\n+++ b/notes.txt\n@@ -0,0 +1 @@\n+hello\n\\\\ No newline at end of file\n\"\"\"\n    files = parse_diff(diff)\n    assert [(f[\"old\"], f[\"new\"]) for f in files] == [(\"shop.py\", \"shop.py\"), (\"/dev/null\", \"notes.txt\")]\n    h1, h2 = files[0][\"hunks\"]\n    assert (h1[\"old_start\"], h1[\"old_count\"], h1[\"new_start\"], h1[\"new_count\"]) == (1, 5, 1, 3)\n    assert h1[\"lines\"][0] == (\"-\", \"def total(prices):\") and h1[\"lines\"][-1] == (\"+\", \"\")\n    assert (h2[\"old_start\"], h2[\"old_count\"], h2[\"new_start\"], h2[\"new_count\"]) == (20, 1, 18, 2)\n    assert files[1][\"hunks\"][0][\"lines\"] == [(\"+\", \"hello\")]")

### Exercise: Which line is that?

Using `parse_diff` (given), write `added_lines(diff)`: a dict from each new file path to the list of `(new_line_number, text)` for its added lines; and `removed_lines(diff)`: the same with old paths and old line numbers for removed lines. Files that were deleted (new path `/dev/null`) have no added lines; files that were added have no removed lines.

In [ ]:
import re

HUNK = re.compile(r"@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")

def _path(p):
    p = p.split("\t")[0].strip()
    return p[2:] if p.startswith(("a/", "b/")) else p

def parse_diff(text):
    files, current, hunk = [], None, None
    for line in text.splitlines():
        if line.startswith("--- "):
            current = {"old": _path(line[4:]), "new": None, "hunks": []}
            files.append(current)
            hunk = None
        elif line.startswith("+++ ") and current is not None and current["new"] is None:
            current["new"] = _path(line[4:])
        elif line.startswith("@@") and current is not None:
            a, b, c, d = HUNK.match(line).groups()
            hunk = {"old_start": int(a), "old_count": int(b) if b is not None else 1,
                    "new_start": int(c), "new_count": int(d) if d is not None else 1, "lines": []}
            current["hunks"].append(hunk)
        elif hunk is not None and line[:1] in (" ", "-", "+"):
            hunk["lines"].append((line[0], line[1:]))
    return files

def added_lines(diff):
    ___

def removed_lines(diff):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_numbers():\n    \"line numbers for comments\"\n    diff = \"--- a/cart.py\\n+++ b/cart.py\\n@@ -10,4 +10,5 @@\\n     def add(self, item, qty=1):\\n-        self.items.append(item)\\n+        for _ in range(qty):\\n+            self.items.append(item)\\n         self.changed = True\\n         return self\\n--- a/old.py\\n+++ /dev/null\\n@@ -1,2 +0,0 @@\\n-x = 1\\n-y = 2\\n\"\n    assert added_lines(diff) == {\"cart.py\": [(11, \"        for _ in range(qty):\"), (12, \"            self.items.append(item)\")]}\n    assert removed_lines(diff) == {\"cart.py\": [(11, \"        self.items.append(item)\")], \"old.py\": [(1, \"x = 1\"), (2, \"y = 2\")]}")

### Exercise: Is this pull request too big?

Write `diff_stats(diff)`: from a unified diff, a dict from each file's path (the new path, or the old one for a deleted file) to `(added, removed)` counts, plus a `"total"` entry. Count `+` and `-` lines inside hunks only (not the `+++` and `---` headers). Then `review_advice(stats, limit=400)`: `"small"` if added plus removed is at most 100, `"fine"` up to `limit`, and otherwise `"split it: N lines"` with the total changed lines.

In [ ]:
def diff_stats(diff):
    ___

def review_advice(stats, limit=400):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stats():\n    \"counting a change\"\n    diff = \"--- a/a.py\\n+++ b/a.py\\n@@ -1,2 +1,3 @@\\n-x = 1\\n+x = 2\\n+y = 3\\n z = 4\\n--- a/b.py\\n+++ /dev/null\\n@@ -1,1 +0,0 @@\\n-gone = True\\n\"\n    stats = diff_stats(diff)\n    assert stats == {\"a.py\": (2, 1), \"b.py\": (0, 1), \"total\": (2, 2)}\n    assert review_advice(stats) == \"small\"\n    assert review_advice({\"total\": (150, 30)}) == \"fine\" and review_advice({\"total\": (390, 60)}) == \"split it: 450 lines\"")

### Exercise: Conventional comments

Write `parse_comment(text)`: a review comment in the Conventional Comments style, `label (decoration, decoration): subject`, becomes a dict with `"label"` (lower case), `"decorations"` (a list, possibly empty), `"subject"` (stripped) and `"blocking"`: true for `issue` and `todo` labels unless decorated `non-blocking`, and for any label decorated `blocking`. Labels are `praise`, `nitpick`, `suggestion`, `issue`, `todo`, `question`, `thought`, `chore` and `note` (`nit` is accepted as `nitpick`). Text that doesn't match raises `ValueError`. Then `must_fix(comments)`: the subjects of the blocking ones, in order.

In [ ]:
import re

LABELS = {"praise", "nitpick", "suggestion", "issue", "todo", "question", "thought", "chore", "note"}

def parse_comment(text):
    ___

def must_fix(comments):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_comments():\n    \"labels and decorations\"\n    assert parse_comment(\"nit: trailing space\") == {\"label\": \"nitpick\", \"decorations\": [], \"subject\": \"trailing space\", \"blocking\": False}\n    assert parse_comment(\"Issue: crashes when the cart is empty\")[\"blocking\"] is True\n    assert parse_comment(\"issue (non-blocking): slow for big carts\")[\"blocking\"] is False\n    assert parse_comment(\"suggestion (blocking, security): escape the name\")[\"decorations\"] == [\"blocking\", \"security\"]\n    assert parse_comment(\"suggestion (blocking, security): escape the name\")[\"blocking\"] is True\n    for bad in [\"this is wrong\", \"rant: why\", \"issue crashes\"]:\n        try:\n            parse_comment(bad)\n            assert False, bad\n        except ValueError:\n            pass\n    comments = [\"praise: lovely tests\", \"issue: off by one in the loop\", \"nit: name\", \"todo: add a changelog entry\", \"question (blocking): is this thread-safe?\"]\n    assert must_fix(comments) == [\"off by one in the loop\", \"add a changelog entry\", \"is this thread-safe?\"]")

### Exercise: Fixing what the review found

A pull request added `summarise(orders)`, and the reviewer left three blocking comments: **issue**: an empty list crashes with `ZeroDivisionError` (it should return `{"count": 0, "total": 0, "average": 0}`); **issue**: refunded orders (with `"refunded": True`) are counted, and they shouldn't be, in any of the figures; **issue**: an exception from a malformed order is swallowed by `except Exception: pass`, hiding bad data (let a `KeyError` from a missing `"amount"` propagate). Fix all three, keeping the rest of the behaviour (the average is rounded to 2 places).

In [ ]:
def summarise(orders):
    total = 0
    count = 0
    for order in orders:
        try:
            total += order["amount"]
            count += 1
        except Exception:
            pass
    return {"count": count, "total": total, "average": round(total / count, 2)}

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_review_fixed():\n    \"all three review comments addressed\"\n    orders = [{\"amount\": 10}, {\"amount\": 25, \"refunded\": True}, {\"amount\": 5.5}, {\"amount\": 4, \"refunded\": False}]\n    assert summarise(orders) == {\"count\": 3, \"total\": 19.5, \"average\": 6.5}\n    assert summarise([]) == {\"count\": 0, \"total\": 0, \"average\": 0}\n    assert summarise([{\"amount\": 3, \"refunded\": True}]) == {\"count\": 0, \"total\": 0, \"average\": 0}\n    try:\n        summarise([{\"amount\": 1}, {\"price\": 2}])\n        assert False, \"a malformed order should raise\"\n    except KeyError:\n        pass\n    assert \"except Exception\" not in source")

### Bug hunt: A hunk count that was left out

In the hunk header `@@ -7 +7,2 @@`, the old count is omitted, which means **1**. The parser uses `0` instead, so it thinks the hunk removes nothing and prints `old lines: 0`. Default a missing count to 1 so it prints `old lines: 1`.

In [ ]:
import re

header = "@@ -7 +7,2 @@ def total(prices):"
m = re.match(r"@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@", header)
old_count = int(m.group(2) or 0)
print("old lines:", old_count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a missing count is one\"\n    expect_output(\"old lines: 1\")")

### Bug hunt: Removing while looping

The reviewer asked: "what happens with two expired items in a row?" The function removes items from the list it's looping over, so the item after each removed one is skipped, and this prints `['milk', 'eggs']` (eggs has expired too). Build a new list instead, so it prints `['milk']`.

In [ ]:
def drop_expired(items):
    for item in items:
        if item["expired"]:
            items.remove(item)
    return [i["name"] for i in items]

stock = [{"name": "bread", "expired": True}, {"name": "eggs", "expired": True}, {"name": "milk", "expired": False}]
print(drop_expired(stock))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"every expired item removed\"\n    expect_output(\"['milk']\")")

### Bug hunt: Counting the headers as changes

The size check counts every line starting with `+` or `-`, including the `+++` and `---` file headers, so a one-line change looks like three, and this prints `added 2, removed 2`. Count only lines inside hunks (after an `@@` line), so it prints `added 1, removed 1`.

In [ ]:
diff = "--- a/x.py\n+++ b/x.py\n@@ -1 +1 @@\n-x = 1\n+x = 2\n"
added = sum(1 for line in diff.splitlines() if line.startswith("+"))
removed = sum(1 for line in diff.splitlines() if line.startswith("-"))
print(f"added {added}, removed {removed}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"headers aren't changes\"\n    expect_output(\"added 1, removed 1\")")

### Puzzle: Reviewing in order

Arrange the steps of reviewing a pull request from the top down. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the line kind that advances both line numbers in a hunk: "+", "-" or " "
q = ___   # the new count in "@@ -3,4 +3 @@"
r = ___   # True or False: style problems a formatter can fix are worth review comments
s = ___   # the label for a comment that must be fixed before merging, in Conventional Comments: "issue" or "nitpick"
t = ___   # roughly how many changed lines a reviewer can review well at once: 40, 400 or 4000

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \" \" and q == 1 and r is False, \"p, q and r\"\n    assert s == \"issue\" and t == 400, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an automatic first-pass reviewer

Write `auto_review(diff)`: check only the **added** lines of a unified diff (as in the second exercise, whose helpers are given) and return a sorted list of `(path, new_line, message)` comments, for `.py` files only:
- `"remove the debugging print"` for an added line containing a call to `print(` outside a string (a line whose stripped text starts with `print(` is enough here), unless the file is under `scripts/`;
- `"bare except hides errors"` for an added line whose stripped text is `except:`;
- `"TODO without an issue number"` for an added line containing `TODO` but no `#` followed by digits (like `#123`);
- `"line too long (N > 100)"` for an added line longer than 100 characters;
- and once per pull request, as `("*", 0, "code changed but no tests changed")`, if any `.py` file outside `tests/` has added or removed lines but no file under `tests/` does.

In [ ]:
import re

HUNK = re.compile(r"@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")

def _path(p):
    p = p.split("\t")[0].strip()
    return p[2:] if p.startswith(("a/", "b/")) else p

def parse_diff(text):
    files, current, hunk = [], None, None
    for line in text.splitlines():
        if line.startswith("--- "):
            current = {"old": _path(line[4:]), "new": None, "hunks": []}
            files.append(current)
            hunk = None
        elif line.startswith("+++ ") and current is not None and current["new"] is None:
            current["new"] = _path(line[4:])
        elif line.startswith("@@") and current is not None:
            a, b, c, d = HUNK.match(line).groups()
            hunk = {"old_start": int(a), "old_count": int(b) if b is not None else 1,
                    "new_start": int(c), "new_count": int(d) if d is not None else 1, "lines": []}
            current["hunks"].append(hunk)
        elif hunk is not None and line[:1] in (" ", "-", "+"):
            hunk["lines"].append((line[0], line[1:]))
    return files

def auto_review(diff):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_reviewer():\n    \"a first pass a machine can do\"\n    long_line = \"x = \" + \"1 + \" * 30 + \"1\"\n    diff = (\n        \"--- a/src/shop.py\\n+++ b/src/shop.py\\n@@ -1,3 +1,7 @@\\n def total(xs):\\n\"\n        \"+    print(xs)\\n+    # TODO handle tax\\n+    # TODO #42 rounding\\n     try:\\n\"\n        f\"+        {long_line}\\n     except:\\n+    except:\\n\"\n        \"--- a/scripts/run.py\\n+++ b/scripts/run.py\\n@@ -1 +1,2 @@\\n x = 1\\n+print(x)\\n\"\n        \"--- a/README.md\\n+++ b/README.md\\n@@ -1 +1,2 @@\\n hi\\n+print(\\n\"\n    )\n    comments = auto_review(diff)\n    assert comments == [\n        (\"*\", 0, \"code changed but no tests changed\"),\n        (\"src/shop.py\", 2, \"remove the debugging print\"),\n        (\"src/shop.py\", 3, \"TODO without an issue number\"),\n        (\"src/shop.py\", 6, f\"line too long ({len(long_line) + 8} > 100)\"),\n        (\"src/shop.py\", 8, \"bare except hides errors\"),\n    ], comments\n    with_tests = diff + \"--- a/tests/test_shop.py\\n+++ b/tests/test_shop.py\\n@@ -1 +1,2 @@\\n x\\n+def test_total():\\n\"\n    assert (\"*\", 0, \"code changed but no tests changed\") not in auto_review(with_tests)")

## Warm-up

That's a **unified diff**, the format of every pull request, `git diff`, and patch file: `-` lines were removed, `+` lines were added, lines with a space are unchanged context, and the `@@` line says where in each version the change sits. Reviewing code mostly means reading this.

## Reading a diff

A unified diff has, for each file:

```text
--- a/src/shop/cart.py
+++ b/src/shop/cart.py
@@ -12,7 +12,8 @@ class Cart:
     def add(self, item, qty=1):
-        self.items.append(item)
+        for _ in range(qty):
+            self.items.append(item)
         self.changed = True
```

- `---` and `+++` name the old and new versions of the file (`/dev/null` for a file that was added or deleted).
- Each **hunk** starts with `@@ -OLD_START,OLD_COUNT +NEW_START,NEW_COUNT @@`, optionally followed by the enclosing function or class as a hint. A count of 1 may be omitted (`@@ -5 +5 @@`).
- Then the lines: ` ` context (in both versions), `-` only in the old, `+` only in the new. Walking the hunk, context and `-` lines advance the old line number; context and `+` lines advance the new one.

Review tools attach comments to a **line of the new file** (or the old one, for a deleted line), so knowing which new line number each `+` line has is exactly what a review tool computes. Python's `difflib.unified_diff` writes this format; `git diff` writes it with a few extra header lines (`diff --git`, `index`).

## What to look for

Review in order, from the top down, because a problem at the top makes the details irrelevant:

1. **Purpose.** Read the description first. What should this change do, and is that the right thing to do? Is it the right size, or several changes in one?
2. **Design.** Does it fit the codebase? Is anything in the wrong place, duplicated or more complicated than it needs to be? Will it be easy to change later?
3. **Correctness.** Does it do what it says? Edge cases: empty, one, many, missing, wrong types, errors half-way, concurrency. Is anything swallowed (`except: pass`)? Read the code paths, not just the happy one.
4. **Tests.** Is new behaviour tested, and would the tests catch the bugs you can imagine (Week 72)? Do they test behaviour, not implementation?
5. **Security and data.** User input reaching queries, files or shells; secrets in code; permissions checked; personal data logged (Week 79).
6. **Readability.** Names, function size, comments, the idioms of yesterday. Could a newcomer follow it?
7. **Details** that tools don't catch. Anything a formatter or linter **can** catch should be caught by them, in CI (Day 4), not by people.

A reviewer can't hold more than about 400 changed lines in their head at once; beyond that, review quality falls off sharply. Small changes get better reviews **and** faster ones.

## Writing comments

Comments are read by a person who worked hard on the change. Good ones are:

- **Kind and about the code**, never the person: "this loop reads the file twice" rather than "you read the file twice".
- **Specific and actionable**: point at the line, say what the problem is, and if you can, suggest a fix (`suggestion:` blocks on GitHub apply in one click).
- **Asking when you're unsure**: "What happens here if `items` is empty?" teaches more, and is less presumptuous, than "this crashes when empty" when you haven't checked.
- **Labelled by importance**, so the author knows what must change. The **Conventional Comments** style starts each comment with a label: `issue:`, `suggestion:`, `question:`, `nitpick:` (or `nit:`), `praise:`, `thought:`, optionally with decorations such as `(blocking)` or `(non-blocking)`: `suggestion (non-blocking): ...`.
- **Generous with praise** where it's earned: "nice: this removes a whole class of bugs" is information too.

Then decide: **approve**, **comment** (no decision), or **request changes** (something must change before merging). Don't block on matters of taste; do block on bugs, missing tests and security problems.

## Being reviewed

Making your change easy to review is your job, not the reviewer's:

- **Keep it small and focused**: one logical change per pull request. Split refactoring from behaviour changes, so each diff is easy to check.
- **Write a description**: what and **why**, how you tested it, anything you're unsure of, and screenshots for anything visual. Link the issue.
- **Review it yourself first.** Read your own diff as a reviewer would: you'll find the debugging `print`, the forgotten file and the confusing name before anyone else does.
- **Make the checks pass** before asking for review.
- **Respond to every comment**: fix it, or reply with your reasoning. Disagreement is fine; resolve it with evidence, or with a quick call when text goes round in circles.

## Practice

## Quiz

## Daily challenge

## Before you go

Open a pull request on any open-source project you use and read its review conversation: notice which comments were labelled, which were questions, and how disagreements were settled. Tomorrow: **documentation**: docstrings that tools can read, doctests that keep examples honest, and documentation sites built from both.